# Лабораторная работа №1
## Освоение базового ML-пайплайна на табличных данных

**Цель работы:** построить воспроизводимый ML-пайплайн для табличного датасета студентов, включающий EDA, проверку качества данных, предобработку, отбор признаков, обучение линейной и логистической регрессии и анализ качества моделей.

На первом этапе выполняются:
- загрузка и первичный осмотр данных;
- проверка пропусков и дубликатов;
- первичная проверка логической корректности данных;
- базовый EDA.


## 1. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)


## 2. Загрузка данных

In [ ]:
df = pd.read_csv('student_lifestyle_100k.csv')
df.head()

## 3. Первичный обзор датасета

Сначала посмотрим размер таблицы, названия столбцов, типы данных и базовые статистики. Это нужно, чтобы понять структуру данных и заранее определить, какие признаки являются числовыми, а какие категориальными.

In [ ]:
print('Размер датасета:', df.shape)
print('\nСтолбцы:')
print(df.columns.tolist())

print('\nТипы данных:')
print(df.dtypes)

print('\nИнформация о датасете:')
df.info()

In [ ]:
df.describe(include='all').T

## 4. Проверка качества данных

Согласно условию лабораторной, необходимо проверить наличие пропусков, дубликатов и логических ошибок. Даже если проблем немного, этот шаг обязателен и должен быть явно показан в ноутбуке.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

print('Количество пропусков по столбцам:')
print(missing)

print('\nКоличество полных дубликатов:', duplicates)

### 4.1 Проверка логических ограничений

В этом датасете особенно важно проверить, не превышает ли суммарное время сна, учёбы, социальных сетей и физической активности 24 часа в сутки. Такие записи не являются техническими ошибками формата, но являются логически сомнительными и могут ухудшать качество модели.

In [ ]:
df['Hours_Sum_3'] = df['Sleep_Duration'] + df['Study_Hours'] + df['Social_Media_Hours']
df['Hours_Sum_4'] = df['Hours_Sum_3'] + df['Physical_Activity'] / 60

logic_checks = {
    'Age < 0': int((df['Age'] < 0).sum()),
    'CGPA outside [0, 10]': int(((df['CGPA'] < 0) | (df['CGPA'] > 10)).sum()),
    'Stress_Level outside [1, 10]': int(((df['Stress_Level'] < 1) | (df['Stress_Level'] > 10)).sum()),
    'Sleep + Study + Social > 24': int((df['Hours_Sum_3'] > 24).sum()),
    'Sleep + Study + Social + Physical/60 > 24': int((df['Hours_Sum_4'] > 24).sum())
}

pd.Series(logic_checks)

### Вывод по качеству данных

На этапе первичной проверки было установлено, что в датасете отсутствуют пропуски и полные дубликаты. Это означает, что основная предобработка будет связана не с заполнением `NaN`, а с логической проверкой данных, анализом выбросов, отбором признаков и корректным построением ML-пайплайна без утечки данных.

Отдельно были проверены физически сомнительные записи, в которых сумма временных затрат за сутки превышает 24 часа. Эти наблюдения будут учтены на следующем этапе предобработки.

## 5. Базовый EDA

На этом этапе изучаются распределения признаков, баланс целевого класса и возможные связи между переменными.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

sns.histplot(df['Age'], bins=20, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Распределение возраста')

sns.histplot(df['CGPA'], bins=25, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Распределение CGPA')

sns.histplot(df['Study_Hours'], bins=25, kde=True, ax=axes[0, 2])
axes[0, 2].set_title('Распределение Study_Hours')

sns.countplot(data=df, x='Gender', ax=axes[1, 0])
axes[1, 0].set_title('Распределение по полу')

sns.countplot(data=df, x='Department', ax=axes[1, 1])
axes[1, 1].set_title('Распределение по Department')
axes[1, 1].tick_params(axis='x', rotation=20)

sns.countplot(data=df, x='Depression', ax=axes[1, 2])
axes[1, 2].set_title('Баланс классов Depression')

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = ['Age', 'CGPA', 'Sleep_Duration', 'Study_Hours', 'Social_Media_Hours', 'Physical_Activity', 'Stress_Level']

plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Корреляционная матрица числовых признаков')
plt.show()

### Краткий вывод по EDA

По результатам первичного анализа видно, что датасет технически хорошо сформирован: признаки имеют ожидаемые типы, целевой столбец `Depression` бинарный, а пропуски отсутствуют. При этом класс `Depression=True` встречается значительно реже, чем `False`, поэтому при обучении классификатора необходимо использовать не только accuracy, но и метрики precision, recall, F1-score и ROC-AUC.

Дополнительно уже на этом этапе можно заметить наличие логических аномалий, связанных с суммарным временем активности за сутки. На следующем этапе будет выполнена аккуратная очистка данных и подготовка признаков к моделированию.